# Phase 2, Task 1 — Meet CICIDS2017

Phase 1 ended with a hard truth: our model caught only ~61% of attacks because it had only ever *seen* a narrow set of attack types. Phase 2 attacks that directly — by moving to a **bigger, more modern, more realistic dataset**: **CICIDS2017** (from the Canadian Institute for Cybersecurity).

## How CICIDS2017 differs from NSL-KDD (why we're switching)
| | NSL-KDD (Phase 1) | CICIDS2017 (Phase 2) |
|---|---|---|
| Age | 1999-era attacks | 2017, modern attacks |
| Attack types | a handful | DDoS, DoS variants, brute force (FTP/SSH), web attacks (XSS, SQLi), infiltration, botnet, port scan |
| Size | ~126k rows | **~2.8 million rows** across 8 daily CSV files |
| Features | 41, hand-crafted | **~78 numeric flow features** (auto-extracted from real packet captures) |
| Cleanliness | pre-cleaned | **messy** — has missing values (NaN) and infinities (Inf) you must handle |
| Balance | roughly even | **very imbalanced** — most traffic is benign, some attacks are tiny slivers |

The messiness and imbalance are *the point* — they're what real security data looks like. Learning to handle them is the skill.

**New terms:**
- **Flow** = one network conversation summarized as numbers (duration, bytes/sec, packet counts, etc.). Same idea as a 'connection' in Phase 1, just far more features.
- **Class imbalance** = when one label (benign) hugely outnumbers another (a rare attack). It quietly breaks naive models — a lazy model can score high 'accuracy' by always guessing benign.
- **NaN / Inf** = 'Not a Number' (missing) and 'Infinity' (e.g. a divide-by-zero in a rate). Most models refuse to train if these are present; we must find and fix them.

## Step 0 — Get the data (do this outside the notebook first)
CICIDS2017 is too big to ship in the repo, so download it once into `data/raw/cicids2017/`.

**Recommended (easiest) — the pre-made CSVs** (folder usually called `MachineLearningCVE`), 8 files, one per day:
```
Monday-WorkingHours.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
... (and the rest)
```
Two free sources:
1. **Official**: https://www.unb.ca/cic/datasets/ids-2017.html (fill the short form → download the `MachineLearningCSV` archive).
2. **Kaggle mirror**: search 'CICIDS2017' — several datasets host the same CSVs.

Put the CSVs in `data/raw/cicids2017/`. You do NOT need all 8 to start — **one file is enough** for this task. A good first pick is the **Wednesday** file (lots of DoS attacks) or **Friday** (DDoS + port scan + botnet).

**When the data is in place, come back and run the cells below.**

## Step 1 — Load ONE day's file and look at it
**Your job:** read one CSV into a DataFrame `df`, then print its `.shape` and `.head()`.

**Hints:**
- `import pandas as pd` then `df = pd.read_csv("../../data/raw/cicids2017/<the file>.csv")`.
- Unlike Phase 1, **this file HAS a header row** — so no `header=None`, no `names=`. pandas reads the column names for you.
- Expect a big shape: hundreds of thousands of rows, ~79 columns.

In [2]:
# TODO: load one CICIDS2017 CSV into df; print df.shape and df.head()
import pandas as pd
df = pd.read_csv("/Users/rohanb/06_projects/CYBERSECURITY/data/raw/CICIDS2017/Wednesday-workingHours.pcap_ISCX.csv")
df

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,389,15206,17,12,3452,6660,1313,0,203.058823,425.778474,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,88,1092,9,6,3150,3152,1575,0,350.000000,694.509719,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
692698,53,32215,4,2,112,152,28,28,28.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
692699,53,324,2,2,84,362,42,42,42.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
692700,58030,82,2,1,31,6,31,0,15.500000,21.920310,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
692701,53,1048635,6,2,192,256,32,32,32.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


## Step 2 — Inspect the columns and the label
**Your job:** see the column names, and find out what attack labels this file contains.

**Hints:**
- `df.columns.tolist()` shows all ~79 feature names. ⚠️ CICIDS2017 column names often have **leading spaces** (e.g. `' Label'`) — note that, it bites people later.
- The target column is usually named `Label` (or `' Label'`). Print `df[' Label'].value_counts()` (adjust the exact name).
- `value_counts()` = counts how many rows have each label. This is where you'll SEE the class imbalance.

In [9]:
# TODO: print df.columns.tolist(), and the value_counts() of the Label column
print(df.columns.tolist())
print(df[' Label'].value_counts())

[' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', 'Total Length of Fwd Packets', ' Total Length of Bwd Packets', ' Fwd Packet Length Max', ' Fwd Packet Length Min', ' Fwd Packet Length Mean', ' Fwd Packet Length Std', 'Bwd Packet Length Max', ' Bwd Packet Length Min', ' Bwd Packet Length Mean', ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min', 'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max', ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std', ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags', ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length', ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s', ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean', ' Packet Length Std', ' Packet Length Variance', 'FIN Flag Count', ' SYN Flag Count', ' RST Flag Count', ' PSH Flag Count', ' ACK Flag Count', ' URG Flag 

## Step 3 — First look at the mess (NaN / Inf)
Real data is dirty. Let's confirm it before we're surprised later.

**Your job:** check whether this file has any missing (NaN) or infinite (Inf) values.

**Hints:**
- Missing count per column: `df.isna().sum()` — then `.sum()` again for a grand total.
- Infinities: `import numpy as np` then `np.isinf(df.select_dtypes('number')).sum().sum()`.
- You don't have to FIX them yet — just find out if they exist. (They usually do.) We'll clean them in the next task.

In [14]:
# TODO: count total NaN values and total Inf values in df
print(df.isna().sum().sum())

import numpy as np
print(np.isinf(df.select_dtypes('number')).sum().sum())

1008
1586


### ✅ You pass Phase 2 Task 1 when:
1. You've downloaded at least one CICIDS2017 CSV into `data/raw/cicids2017/` and loaded it into `df`.
2. You can state the file's shape and name 3 differences between this data and NSL-KDD.
3. You've printed the `Label` value_counts and can point to the **class imbalance** (which label dominates?).
4. You know whether this file contains NaN and/or Inf values (yes/no + rough count).

Paste me your shape, your Label value_counts, and your NaN/Inf totals — then Task 2 is cleaning this data and getting it model-ready.